In [1]:
%run ../../Latex_macros.ipynb

<IPython.core.display.Latex object>

# Session 7: Policy gradient III: PPO and GRPO

## Where this session fits

- **Series goal**
    - choose good actions by trial and error
    - from small tables to robots and language models
- **Position**
    - session 7: last session made the policy gradient practical with a baseline, the advantage, REINFORCE and the surrogate loss
    - this session adds the two methods used to train language models: PPO and GRPO
    - the next session applies them to language models
- **This session's goal**
    - see how a clipped ratio keeps each update small, and how GRPO drops the critic
- **By the end you can**
    - explain the probability ratio (an importance weight) and what clipping does
    - say how GAE gives per-step credit from a terminal reward
    - read the PPO pseudo-code: actor, critic, old actor and reference model
    - compute the GRPO advantage for a group of completions


# PPO (the "Classic")


## Intuition

PPO optimizes a **clipped surrogate** loss
- which can be interpreted as variant of the standard policy gradient theorem.

The goal of the surrogate objective is to control how rapidly the policy changes from iteration to iteration of training.
- avoid large policy shifts
- smaller, more stable updates
- (a monotonic-improvement guarantee belongs to the theory of TRPO, not to PPO's clipping)

## Surrogate Loss for PPO

The Surrogate Loss for PPO is
$$
L_{\text{clip}}(\theta) = \E_\tt \left[
\min{} \left(
r_\tt(\theta) \hat{\advseq}_\tt,\;
\text{clip}\left(r_\tt(\theta), 1 - \epsilon_{\text{clip}}, 1 + \epsilon_{\text{clip}}\right) \hat{\advseq}_\tt
\right)
\right]
$$

We will explain how this potentially intimidating equation came about.

- this objective is *maximized*; the code below minimizes its negative
- $\E_\tt$ is the average over the time steps of the batch; $\text{clip}(x, a, b)$ limits $x$ to the interval $[a, b]$; $r_\tt$ and $\epsilon_{\text{clip}}$ are defined next


We express the 
- relative change in policy (for a given State/Action pair)
    - of the current Actor
    - compared to the Old Actor (Actor at start of current iteration)
- via the *probability ratio*
$$
r_\tt(\theta) = \frac{\pi_\theta(\actseq_\tt |\stateseq_\tt )}{\pi_{\theta_{\text{old}}}(\actseq_\tt |\stateseq_\tt)}
$$

where $\pi_{\theta_{\text{old}}}$ is the *Old Actor policy*

**Notation note:** the PPO paper writes the probability ratio as $r_\tt(\theta)$, and we keep it. This $r$ is not the reward (written $\rew$ or $\rewseq_{\tt+1}$ here); context tells them apart.

The ratio is also an *importance weight*: the episodes come from the old actor, but we are evaluating the new one.


The Old Actor policy is the policy in effect at the start of each iteration of training
- before this iteration modifies the policy parameters
- it is the policy *under which examples for the current batch* have been generated
    - the *Behavior Policy* in On-Policy/Off-Policy terminology

During each iteration of training
- episodes/trajectories are generated  using the *Old Actor* policy
    - stored in a buffer for the iteration
    - are used *multiple times* during the iteration
- the policy parameters are updated based on the surrogate loss for these trajectories
- the updated policy becomes the Old Actor policy for the next iteration

Clipping discourages large changes in the policy (it does not strictly bound the ratio).


How far can the ratio go?
- the *clip* range is $[1 - \epsilon_{\text{clip}}, 1 + \epsilon_{\text{clip}} ]$ for a small $\epsilon_{\text{clip}}$ (typically $0.2$)
- clipping does not force $r_\tt$ to stay in the range: it removes the gradient of a sample once its ratio has moved *outside* the range in the direction the advantage favours

this is the *clipped* part of the clipped surrogate objective.

The PPO paper writes the clip range as $\epsilon$; we add the subscript, $\epsilon_{\text{clip}}$, to distinguish it from the exploration probability $\epsilon$ of $\epsilon$-greedy methods.

Formally

The *surrogate loss* in PPO is:

$$
L_{\text{sur}}(\theta) = \E_\tt \left[ r_\tt(\theta) \hat{\advseq}_\tt \right] 
= \E_\tt \left[ \frac{\pi_\theta(\actseq_\tt|\stateseq_\tt)}{\pi_{\theta_{\text{old}}}(\actseq_\tt|\stateseq_\tt)} \hat{\advseq}_\tt \right]
$$

where

- $ r_\tt(\theta)$  is the probability ratio,
- $\hat{\advseq}_\tt$ is the advantage estimate at step $\tt$.



We modify the Surrogate Loss to limit the effect of the Probability Ratio.

The *clipped surrogate objective* (to be maximized) is

$$
L_{\text{clip}}(\theta) = \E_\tt \left[ \min{} \left( r_\tt(\theta) \hat{\advseq}_\tt, \; \text{clip}\left(r_\tt(\theta), 1 - \epsilon_{\text{clip}}, 1 + \epsilon_{\text{clip}} \right) \hat{\advseq}_\tt \right) \right]
$$

We maximize it (Gradient Ascent), in a controlled manner
- equivalently, we minimize its negative, which is what the code below calls `L_clip`



**What clipping does**

<img src="images/ppo_clip.png" width=95%>

- $r > 1$ means the action became more likely than under the old actor; $r < 1$, less likely
- with a positive advantage the objective stops rewarding an increase of $r$ beyond $1 + \epsilon_{\text{clip}}$; with a negative advantage, a decrease below $1 - \epsilon_{\text{clip}}$
- where the curve is flat the gradient is zero, so the update stops pushing that action further
- the `min` keeps the other two cases live: a ratio that has moved the *wrong* way (sloped branches) still gets pushed back


## Iterations vs Epochs

In Supervised Learning an *epoch* processes all the examples (the targets are fixed).

In Reinforcement Learning the examples are episodes, generated by the current actor
- an *iteration* generates a batch of episodes with the actor as it is at the start of the iteration
- then repeats **epochs** (passes) over that fixed batch, updating the actor and the critic each epoch
- why repeat? sample efficiency: a long episode is expensive, so we learn as much as possible before discarding it
- the first epoch is exactly on-policy ($r_\tt = 1$); later epochs are slightly off-policy, corrected by the ratio


## Relation to the Unified Gradient Formulation

The (unclipped) Surrogate Loss does not resemble the  Unified Gradient Formulation.
- absence of the term

$$
\log \pi_\theta(\actseq_{\tau,\tt} | \stateseq_{\tau,t}) 
$$

that multiplies the Advantage.

Moreover,
 there is an extra term
$$
r_\tt(\theta) = \frac{\pi_\theta(\actseq_\tt |\stateseq_\tt )}{\pi_{\theta_{\text{old}}}(\actseq_\tt |\stateseq_\tt)}
$$

in the form of the Probability Ratio.


But, these terms appear inside the Expectation of the Gradient and, by using the Likelihood Ratio trick
$$
\begin{array}{llll} \\
\nabla_\theta r_\tt(\theta) & = & r_\tt (\theta) \nabla_\theta \log r_\tt (\theta) & \text{Likelihood Ratio trick} \\
& = & r_\tt (\theta) \nabla_\theta \left( 
    \log \pi_\theta(\actseq_\tt|\stateseq_\tt) - \log \pi_{\text{old}} (\actseq_\tt|\stateseq_\tt) 
\right)  & \text{log of ratio converted to difference in logs} \\
& = & r_\tt (\theta) \nabla_\theta \log \pi_\theta(\actseq_\tt|\stateseq_\tt) & \text{since } \log \pi_{\text{old}} \text{ is not a function of } \theta \\
\end{array}
$$

Thus, taking the gradient of the Probability Ratio
- yields the usual
$$
\log \pi_\theta(\actseq_{\tau,\tt} | \stateseq_{\tau,t}) 
$$

term.

At $r_\tt = 1$ (the first epoch, when $\theta = \theta_{\text{old}}$) the gradient of the surrogate is exactly the gradient of the Unified Gradient Formulation.

In later epochs the extra factor $r_\tt(\theta)$
- is the importance weight that corrects for the data being slightly old
- and clipping removes the gradient of samples whose ratio has moved too far
- the ratio is a correction *per action*: it ignores the shift in which states are visited


## Advantage for PPO

To complete the presentation in terms of the Unified Gradient formulation, we define the Advantage.

| Reward Setting         | Advantage Estimate                                  |
|:-----------------------|:-----------------------------------------------------|
| Intermediate Rewards  | $\hat{\advseq}_\tt = \rlret_\tt -  \statevalfun_\pi(\stateseq_\tt)$ or GAE                 |
| Single Total Reward   | GAE: the critic spreads the single reward over the steps through $\delta_\tt$ |



We need to do a "deeper dive" on PPO in order to truly understand (and appreciate)
the Advantage definition.

In short
- PPO uses a Critic NN to estimate a Value function.e.g., the
$$
\statevalfun_\pi(\stateseq_\tt)
$$
term in the table above

and, in GAE, the TD error (here called the *local surprise*)
$$
\delta_{\tt}
$$
in the GAE description below.


**Using Return-to-go and value estimate:**

$
\hat{\advseq}_\tt = \rlret_\tt - \statevalfun_\pi(\stateseq_\tt)
$

where 
- $\rlret_\tt$ is the *actual* return-to-go starting from time step $\tt$
    - in the trajectory that was encountered
- $\statevalfun_\pi(\stateseq_\tt)$ is the value of $\rlret_\tt$ that was anticipated by the Value function

That is $\hat{\advseq}_\tt$ 
- is a measure of the error between actual fact and previously assumed value.



We can use the TD error $\delta_\tt$ of the value-based sessions, here called the *local surprise*
- as an *attribution of the error*

to each step in the trajectory

$
\hat{\advseq}_\tt = \sum_{l=0}^{\infty} (\disc \lambda)^l \delta_{\tt+l}
$

where

$
\delta_\tt = \rewseq_{\tt+1} + \disc \statevalfun_\pi(\stateseq_{\tt+1}) - \statevalfun_\pi(\stateseq_\tt)
$

with

- $\disc$ as the discount factor,
- $\lambda \in [0, 1]$ controlling the bias-variance trade-off.

**GAE with a single terminal reward**

- the critic gives values $\statevalfun(\stateseq_\tt)$; the value of a terminal state is $0$ and the sum stops at the end of the episode
- with a reward only at the last step, $\delta_\tt = \disc \, \statevalfun(\stateseq_{\tt+1}) - \statevalfun(\stateseq_\tt)$ at every earlier step, and $\delta_\tt = \rewseq_{\tt+1} - \statevalfun(\stateseq_\tt)$ at the last
- so every step gets credit through *how the critic's value changed*: the single reward is spread over the steps by the critic's predictions
- this is **GAE** (Schulman et al., 2015); $\lambda$ sets how many steps of $\delta$ are added


**Aside**

- When $\lambda = 0$, GAE reduces to the **1-step Temporal Difference (TD) advantage** $\hat{\advseq}_\tt = \delta_\tt$:

- When $\lambda = 1$, GAE reduces to the **Monte Carlo advantage** $\hat{\advseq}_\tt = \rlret_\tt - \statevalfun_\pi(\stateseq_\tt)$:

GAE smoothly interpolates between 
- high-bias/low-variance and low-bias/high-variance advantage estimates

making it very effective in practice.

## Pseudo code for PPO

This is RLHF-style PPO: the reference model and the KL term are not in the PPO paper. There are many subtleties to the code, which we discuss below:

    # --- INITIALIZATION ---
    # pi_theta: Training Actor (weights: theta)
    # v_phi:    Training Critic (weights: phi)
    # pi_old:   Rollout Snapshot (weights: theta_old)
    # pi_ref:   Safety Anchor (weights: frozen)

    for iteration = 1, 2, ... do:

        # -------------------------------------------------------------------------
        # 1. ROLLOUT PHASE (Sampling)
        # -------------------------------------------------------------------------
        # ACTION BY: [ACTOR (pi_theta)]
        # We generate trajectories using the live model weights
        actions, states, rewards = Rollout(pi_theta)   # pi_theta equals pi_old here: pi_old is synchronized at the end of every iteration

        # -------------------------------------------------------------------------
        # 2. PRE-COMPUTATION (Frozen Stats for Optimization)
        # -------------------------------------------------------------------------
        with torch.no_grad():
            # CALCULATED BY: [OLD ACTOR (pi_old)]
            # Anchor the original action probabilities for the ratio calculation
            logp_old = pi_old(actions | states)

            # CALCULATED BY: [CRITIC (v_phi)]
            # Get baseline "expected rewards" to center the training signal
            values_old = v_phi(states)

            # CALCULATED BY: [REFERENCE (pi_ref)]
            # Establish the "standard" behavior probability to check for drift
            logp_ref = pi_ref(actions | states)

            # [ADVANTAGE CALCULATION]
            # Combines Rollout rewards (Env) and Critic predictions (Baseline)
            # Goal: Reduce Environment Variance
            A_hat = Compute_GAE(rewards, values_old)
            Returns = A_hat + values_old

        # -------------------------------------------------------------------------
        # 3. OPTIMIZATION PHASE (Learning)
        # -------------------------------------------------------------------------
        for epoch = 1 to K do:     # K is typically 3 to 10; eps (the clip range) is typically 0.2

            # CALCULATED BY: [ACTOR (pi_theta)]
            # Get logprobs of the SAME actions under CURRENTLY UPDATING weights
            logp_curr = pi_theta(actions | states)

            # CALCULATED BY: [CRITIC (v_phi)]
            # Get current value predictions to compare against actual Returns
            v_curr = v_phi(states)

            # --- LOSS CALCULATION ---

            # [ACTOR LOSS] - Uses pi_theta (logp_curr) & pi_old (logp_old) & Critic (A_hat)
            ratio = exp(logp_curr - logp_old)
            L_clip = -min(ratio * A_hat, clip(ratio, 1-eps, 1+eps) * A_hat)

            # [KL PENALTY] - Uses pi_theta (logp_curr) & pi_ref (logp_ref)
            d = logp_ref - logp_curr
            L_kl = beta * (exp(d) - d - 1)   # non-negative estimator of the KL value (practices differ on how to differentiate it)

            # [CRITIC LOSS] - Uses v_phi (v_curr) & actual trajectory Returns
            L_value = c1 * MSE(v_curr, Returns)

            # [UPDATE]
            # Calculate gradients for theta and phi
            Total_Loss = L_clip + L_kl + L_value
            Update theta, phi via SGD

        # -------------------------------------------------------------------------
        # 4. SYNCHRONIZATION
        # -------------------------------------------------------------------------
        # Update pi_old for the next iteration's ratio anchor
        pi_old.load_state_dict(pi_theta.state_dict())

### Key observations about the code

**Iterations vs epochs**

- Examples for the iteration are created at the start of the iteration
- The model that creates the examples
    - changes at the end of the iteration
    - **not** within the iteration
- the Actor and Critic are updated
    - **within each epoch** of **each iteration**
- the Actor **does not** create new episodes using the updated policy
    - it uses the episodes created at start of iteration
    - evaluates *new probabilities* on the states/actions of the start-of-iteration episode

**Multiple models are computing values: be aware of which does what**        

Four models appear in the code: the actor $\pi_\theta$, the critic $v_\phi$, the old actor $\pi_{\text{old}}$ and a frozen reference model $\pi_{\text{ref}}$ (the last slide of this section)


There is **one set of examples** per iteration
- ROLLOUT phase
    - creates examples that are **fixed** for this iteration
    - created by the Actor from end of previous iteration
        - this Actor is called *Old Actor* subsequent to the ROLLOUT phase
    - *Old Actor* is the **Behavior Policy** (in On-policy/Off-policy terminology)
      
- PRE-COMPUTATION phase
    - we now call the model that create the examples the OLD ACTOR
    - the Advantage is computed with respect to the OLD ACTOR
        - using rewards and states from the episodes (examples) created by OLD ACTOR
        - NOT the rewards, states generated by the current (updated) ACTOR model
        - Using the Critic as it was at the end of the previous *iteration*
            - (`values_old` is computed once per iteration, before the epoch loop)
            - to compute state values

- OPTIMIZATION phase
    - the episodes remain the ones generated by the OLD ACTOR
        - **not** using updated policy of the ACTOR
        - so the states/actions/rewards are the same
        - but the *probabilities* are updated
            - used in the ratio

**Loss function**

Contains multiple terms
- `L_clip` and `L_kl` are part of Actor Loss
    - updates Actor parameters $\theta$
- `L-value` are part of Critic Loss (MSE)
    - updates Critic parameters $\phi$
    
So both Actor and Critic
- are updated each *epoch*

The Old Actor
- changes at end of *iteration*
    - to the Actor model as updated at end of the iteration

**The reference model and the KL term**

- $\pi_{\text{ref}}$ is a frozen copy of the model we started from (for a language model: the supervised fine-tuned model)
- the KL penalty discourages $\pi_\theta$ from drifting far from it
- it comes from RLHF practice and is **not** part of the PPO paper (which uses clipping, or an adaptive KL penalty against the *old* policy)
- the plain log-ratio $\log \pi_\theta - \log \pi_{\text{ref}}$ has zero expected gradient on samples from $\pi_\theta$ (the zero-mean score), so the code uses the non-negative estimator $e^{d} - d - 1$ with $d = \log \pi_{\text{ref}} - \log \pi_\theta$ (it estimates the KL *value*; its naive gradient is that of the *reverse* KL, so how best to differentiate the KL is still debated)
- classic RLHF puts a per-token KL into the *reward* instead; GRPO puts the KL in the loss


## Where clipping comes from: Trust Region Policy Optimization (TRPO)

Constraining the probability ratio
$$
r_\tt(\theta) = \frac{\pi_\theta(\actseq_\tt |\stateseq_\tt )}{\pi_{\theta_{\text{old}}}(\actseq_\tt |\stateseq_\tt)}
$$
to the small range $$[1 - \epsilon_{\text{clip}}, 1 + \epsilon_{\text{clip}}]$$

- limits how far one batch of episodes can move the policy
- (TRPO's theory gives a monotonic-improvement bound for its exact constrained problem; clipping is only a practical approximation of that idea)

Constraining the "Actor" versus "Old" policy is based on
 *Trust Region Policy Optimizaton (TRPO)*.

We state TRPO briefly.

TRPO 
- limits the change between two policies
- "Old" and "Actor" (current)

under the assumption that we "trust" the "Old"
- it is the Behavior Policy that generated the examples

It *constrains* the optimization
- to maximize a surrogate objective 

$$
\max{\theta} \quad \E_{\state,\act \sim \pi_{\theta_{\text{old}}}} \left[
    \frac{\pi_\theta(\act \mid \state)}{\pi_{\theta_{\text{old}}}(\act \mid \state)} \advseq^{\pi_{\theta_{\text{old}}}}(\state,\act)
\right]
$$
- subject to a constraint 
    - on how much the new policy can deviate from the old policy:
    $$
\E_{\state \sim \pi_{\theta_{\text{old}}}} \left[
    D_{\text{KL}}\big( \pi_{\theta_{\text{old}}}(\cdot \mid \state) \,\|\, \pi_\theta(\cdot \mid \state) \big)
\right] \leq \delta_{\text{KL}}
$$

where:

- $\pi_{\theta_{\text{old}}}$ is the old  policy,
- $\pi_\theta$ is the new policy parameterized by $\theta$,
- $\advseq^{\pi_{\theta_{\text{old}}}}(\state,\act)$ is the advantage function with respect to the old policy,
- $D_{\text{KL}}(\cdot \| \cdot)$ is the Kullback-Leibler (KL) divergence measuring the difference between the old and new policies,
- $\delta_{\text{KL}}$ is a small positive constant controlling the maximum allowed policy update
    - the TRPO paper writes this bound as $\delta$; the subscript distinguishes it from the TD error $\delta_\tt$

With regards to the "trust region":

TRPO
- uses the KL divergence term as a constraint

PPO
- uses the *clipping* of the ratio term to discourage large changes

PPO is a heuristic in the spirit of TRPO
- it replaces the hard KL constraint with a clipped probability ratio
- without TRPO's guarantee: the ratio can still leave the range



**Technical note**

The first argument in KL term of  TRPO
- is $\pi_\text{old}$
- **not** $\pi_\theta$

The KL function is *asymmetric*
- the expectation is taken over $\pi_\text{old}$ not $\pi_\theta$

This is because "Old" is the behavior policy -- it is our "empirical distribution" of training
- any action introduced by the updated policy
- which had $0$ probability in "Old"
- *has no direct effect* on the KL term
    - since it has $0$ weight
    - only an indirect effect by shifting probability from elsewhere

# GRPO (the "new kid")

GRPO keeps PPO's clipped ratio but removes the critic
- sample a group of $G$ completions $\tau_1, \ldots, \tau_G$ for each prompt; completion $i$ has return $\rlret_{0,\tau_i}$ (written $r_i$ in the papers)
- advantage of completion $i$: its return minus the group mean, divided by the group standard deviation; the *same* advantage for every token of the completion
- objective: the clipped ratio times the advantage, averaged over tokens and completions, minus a KL penalty to a reference model
$$
\frac{1}{G} \sum_{i=1}^{G} \frac{1}{|\tau_i|} \sum_{\tt} \min{} \bigl( r_{i,\tt}(\theta) \hat{\advseq}_i, \; \text{clip}\bigl(r_{i,\tt}(\theta), 1 - \epsilon_{\text{clip}}, 1 + \epsilon_{\text{clip}}\bigr) \hat{\advseq}_i \bigr) - \beta \, \text{KL}
$$
- if all $G$ returns are equal the advantages are $0$ (no gradient; a small constant guards the division)
- two subtleties: the group mean contains the completion itself, which scales the expected gradient by $(G-1)/G$ (the leave-one-out mean avoids this); dividing by the standard deviation changes how prompts are weighted
- GRPO is *not* a preference method: unlike DPO it needs no pairs of preferred and rejected responses, only a score for each completion
- papers: PPO (Schulman et al., 2017), TRPO (2015), GAE (2015), DeepSeekMath (2024)


**GRPO by numbers**

- one prompt, $G = 4$ completions, rewards $(1, 0, 0, 0)$
- mean $= 0.25$, standard deviation $= 0.5$ (sample standard deviation)
- advantages: $(1 - 0.25)/0.5 = 1.5$ for the successful completion and $(0 - 0.25)/0.5 = -0.5$ for each of the others
- if all four rewards are $1$ (or all $0$), every advantage is $0$: that prompt gives no gradient

        for prompt in batch:
            completions = sample G completions with the old policy
            r = [reward(c) for c in completions]
            A = (r - mean(r)) / (std(r) + 1e-8)          # one number per completion
            for epoch in 1..K:
                ratio = pi_theta(token) / pi_old(token)  # per token
                loss  = -mean(min(ratio * A, clip(ratio, 1 - eps, 1 + eps) * A)) + beta * KL
                update theta


# What we established, and what it sets up

- **Established**
    - PPO weights by a critic-based advantage (GAE) and clips the probability ratio, so no single update moves the policy far
    - the PPO pseudo-code has four models: actor, critic, old actor and (for language models) a frozen reference model with a KL penalty
    - GRPO keeps the clipped ratio, drops the critic, and scores each completion against its own group
- **Sets up**
    - next session: the language model is the policy, a response is an episode, and GRPO needs no critic


In [2]:
print("Done")

Done
